In [ ]:
import os
from pathlib import Path
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance, KeyBERTInspired, VisualRepresentation
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from PIL import Image
import base64
from io import BytesIO
from IPython.display import HTML

# Disable parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Load model if script is done
#topic_model = BERTopic.load("model")


In [ ]:
# Step 0: Load Images and Captions
captions_csv = Path("Results/captions.csv")
captions = pd.read_csv(captions_csv)[['image', 'caption']]

# Extract image filenames and captions
image_folder_path = "Data"
image_filenames = captions['image'].tolist()
docs = captions['caption'].tolist()
images = [str(Path(image_folder_path) / filename) for filename in image_filenames]


In [ ]:
# Step 1: Load Embedding Model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Step 2: Define Vectorizer and CTFIDF Model
vectorizer_model = CountVectorizer(stop_words="english")
ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)


In [ ]:
# Step 3: Define Representation Model
representation_model = {
    "Visual_Aspect": VisualRepresentation(nr_repr_images=9),
    "KeyBERT": KeyBERTInspired(),
    "MMR": MaximalMarginalRelevance(diversity=0.3)
}

In [ ]:
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic representations
)

In [ ]:
# Step 5: Run Model

topics, probs = topic_model.fit_transform(documents=docs, images=images)


In [ ]:
# Step 6: Save the Model 

topic_model.save("my_model", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
def image_base64(im):
    # If `im` is a string (path), load the image
    if isinstance(im, str):
        im = Image.open(im)
    
    # Convert the image to base64
    with BytesIO() as buffer:
        im.save(buffer, 'JPEG')  # Save as JPEG, or 'PNG' if you need that format
        return base64.b64encode(buffer.getvalue()).decode()

def image_formatter(im):
    # Return the image HTML code with the base64 encoding
    return f'<img src="data:image/jpeg;base64,{image_base64(im)}" />'

# Extract dataframe with correct syntax for dropping columns
df = topic_model.get_topic_info().drop("Representative_Docs", axis=1).drop("Name", axis=1)

# Visualize the images (assuming `Visual_Aspect` column contains paths or image objects)
HTML(df.to_html(formatters={'Visual_Aspect': image_formatter}, escape=False))

In [ ]:
# Step 8: Save Results to CSV
results = pd.DataFrame({"Image": images, "Topic": topics})
results.to_csv("Results/topic_model_results.csv", index=False)